<a href="https://colab.research.google.com/github/prometheus404/NLP_proj/blob/master/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NLP project

In [1]:
%pip install llama-cpp-python==0.2.90 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu122


In [2]:
from llama_cpp import Llama, llama_free, llama_free_model
from tqdm import tqdm
#from transformers import AutoTokenizer, pipeline, BitsAndBytesConfig
import requests
from collections import defaultdict
import json
import torch


In [3]:
# Load the model
chosen = 'llama'
models = {
    'llama': {'repo_id':"bartowski/Meta-Llama-3.1-8B-Instruct-GGUF",
              'filename':"Meta-Llama-3.1-8B-Instruct-Q8_0.gguf"},
    'qwen': {'repo_id':"Qwen/Qwen3-8B-GGUF",
             'filename': "Qwen3-8B-Q8_0.gguf"},
    'mistral': {'repo_id':"TheBloke/Mistral-7B-v0.1-GGUF",
                'filename':"mistral-7b-v0.1.Q8_0.gguf"},
}

model = Llama.from_pretrained(repo_id=models[chosen]['repo_id'], # repository name
                            filename=models[chosen]['filename'], # model file
                            n_gpu_layers=-1, # use all GPU layers
                            n_ctx=32768, # context size
                            flash_attn=True, # use flash attention
                            chat_format="llama-3", # chat format
                            verbose=False)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
rulebook = requests.get('https://raw.githubusercontent.com/prometheus404/NLP_proj/refs/heads/master/rules/texts/dominion.txt').text
rulebook[:100]

'# Dominion\nYou are a monarch, like your parents before you - a ruler of a small pleasant kingdom of '

In [5]:
# Check that input is inside context window
#tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1")

#tokens = tokenizer.encode(rulebook)
#print(len(tokens))

In [6]:
def generate_message(prompt, rulebook):
    return [
        {
                "role": "system",
                "content": prompt,
            },
            {
                "role": "user",
                "content": "Here is the rulebook:\n"+rulebook,
            },
    ]

In [7]:
import gc
def multiple_model_test(prompts, file_names, iterations, test_name):
    outputs = defaultdict(dict)

    for game,prompt_name,prompt,it in tqdm([(f,pn,p,it) for f in file_names for (pn,p) in prompts for it in range(iterations)]):
        rulebook = requests.get('https://raw.githubusercontent.com/prometheus404/NLP_proj/refs/heads/master/rules/texts/'+game+'.txt').text
        out = model.create_chat_completion(generate_message(prompt, rulebook), temperature=0.7)
        outputs[chosen+'-'+game+'-'+prompt_name][str(it)] = out['choices'][0]['message']['content']

    with open(f'{test_name}.out','w') as f:
        json.dump(dict(outputs),f)

    return outputs

# Rule extraction
1. Give the model a rulebook and prompt it to explain the game in simple, conversational terms to a child or other audiences. -> tree decomposition to test how well the model did
2. Test the ability of the model to find analogies of rules (?)
3. Test the ability to extract if-then rules (?)
4. Organize the rules of into a hierarchy: top-level objectives, mid-level phases, low-level actions. (?) (look into the paper)

In [8]:
prompts = [("kid", """You are a friendly tutor explaining board games to a 7‑year‑old. Summarize the game in plain language, using short sentences and with fun tone. Include:
                - Goal of the game
                - How a player wins
                - What a turn looks like
                - Exceptions to standard rules

                The user will give you a text file with the rulebook you need to explain.
                the output should not be too long. All rules must be present in your explanation"""),
           ("analogies", """The user will give you a text file with the rulebook you need to explain
           the output should not be too long. All rules must be present in your explanation.
           Explain the rules to a child by comparing it to something they already know (e.g. some other famous board games).
           se the rulebook to keep the analogy accurate, and end with a one‑sentence “what you try to achieve” statement.\n"""),
]
game_names = [ 'dominion','7_wonders', 'catan', 'power_grid','ticket_to_ride',]

iterations = 1

multiple_model_test(prompts, game_names, iterations, 'extraction')

100%|██████████| 10/10 [03:00<00:00, 18.05s/it]


defaultdict(dict,
            {'llama-dominion-kid': {'0': "Let me explain Dominion in a way that's easy to understand!\n\n**What's the goal of the game?**\nThe goal of Dominion is to build the best deck of cards by collecting and playing cards that will give you victory points. The player with the most victory points at the end of the game wins!\n\n**How do you win?**\nYou win by having the most victory points at the end of the game. Victory points are represented by a symbol called the <shield>. You get victory points from certain cards, like Provinces or Gardens.\n\n**What happens on a turn?**\nA turn is divided into three phases: Action, Buy, and Clean-up.\n\n1. **Action Phase**: You can play one Action card from your hand. Action cards have special instructions, like drawing cards or gaining coins. You can play multiple Action cards if you have the right cards.\n2. **Buy Phase**: You can play Treasure cards to get coins, and then use those coins to buy a new card from the Supply. 

## Error detection

 Each rulebook is edited by inserting a set of 5 errors each of increasing difficulty:
 - level 1: **missing** -> an entire paragraph of the rulebook describing some core mechanic is missing
- level 2: **unsolvable** -> (in one line states you can draw two cards, in another one that you can draw only one)
- level 3: **incoherent** -> a mechanic that hardlocks the game (you cannot play train if you do not have train on the map but on another line clearly states you start with an empty map)
- level 4: **gamebreaking** -> a coherent but obviously gamebreaking mechanic (whenever you draw a card you can draw another card)
- level 5: **unbalanced** -> a coherent but very unbalanced mechanic (the first player can play two turns)

In [9]:
prompts = ["""test"""]
file_names = ['ticket_to_ride', 'dominion', 'catan', 'power_grid']
models = [llm]
iterations = 5

multiple_model_test(models, prompts, file_names, iterations, 'error_detection')

NameError: name 'llm' is not defined

# Game classification
Give the model a rulebook and ask it to classify the mechanics, evaluate the complexity, suggests the perfect number of players and estimate the duration